# Exercitiul 6 - Inferență bayesiană Poisson–Gamma

O companie de telecomunicatii vrea sa estimeze rata medie $\lambda$ a apelurilor primite intr-o ora. In 10 ore au fost primite 180 de apeluri. Numărul de apeluri pe oră este modelat Poisson($\lambda$), iar pentru $\lambda$ se folosește un prior Gamma.

Se cer: distribuția a posteriori, intervalul HDI de 94% și cea mai probabilă valoare a lui $\lambda$.


## Alegerea priorului

Folosim media istorică $\mu=15$ și deviația standard $\sigma\approx 9$:

$$\alpha_0=\left(\frac{\mu}{\sigma}\right)^2\approx 2.78,\qquad \beta_0=\frac{\mu}{\sigma^2}\approx 0.185.$$

Rotunjim la $\alpha_0=3$ și $\beta_0=0.2$, deci priorul este $\text{Gamma}(3, 0.2)$.


In [7]:
from scipy.stats import gamma
from scipy.optimize import brentq

alpha0, beta0 = 3.0, 0.2
n_ore, nr_apeluri = 10, 180

alpha_n = alpha0 + nr_apeluri
beta_n = beta0 + n_ore
mod = (alpha_n - 1) / beta_n


def hdi_gamma(alpha, beta, prob=0.94):
    scale = 1.0 / beta

    def diferenta(h):
        l = gamma.ppf(gamma.cdf(h, alpha, scale=scale) - prob, alpha, scale=scale)
        if l <= 0:
            return -gamma.pdf(h, alpha, scale=scale)
        return gamma.pdf(l, alpha, scale=scale) - gamma.pdf(h, alpha, scale=scale)

    h_min = gamma.ppf(prob, alpha, scale=scale)
    h_max = gamma.ppf(1 - 1e-12, alpha, scale=scale)
    h = brentq(diferenta, h_min, h_max)
    l = gamma.ppf(gamma.cdf(h, alpha, scale=scale) - prob, alpha, scale=scale)
    return l, h


hdi = hdi_gamma(alpha_n, beta_n, 0.94)

print(f'Posterior: Gamma({alpha_n:g}, {beta_n:g})')
print(f'HDI 94%: [{hdi[0]:.2f}, {hdi[1]:.2f}]')
print(f'Valoarea cea mai probabilă: {mod:.2f}')


Posterior: Gamma(183, 10.2)
HDI 94%: [15.47, 20.45]
Valoarea cea mai probabilă: 17.84


## Rezultate

- **Posterior:** $\lambda \mid y \sim \text{Gamma}(183,\; 10.2)$
- **HDI 94%:** aproximativ $[15.5,\; 20.4]$ apeluri/oră
- **Valoarea cea mai probabilă:** $182/10.2 \approx 17.84$ apeluri/oră
